# Y Combinator Startup Directory Scraper

## 📋 Assignment Overview
This notebook scrapes data for approximately **500 startups** from the [Y Combinator startup directory](https://www.ycombinator.com/companies).

### Data Extracted:
1. Company Name
2. Batch
3. Short Description
4. Founder Name(s)
5. Founder LinkedIn URL(s)

### Approach:
1. **API Discovery**: YC uses Algolia for their search functionality - we leverage this for efficient batch fetching
2. **Page Scraping**: Individual company pages are scraped for detailed founder information
3. **Concurrent Processing**: ThreadPoolExecutor enables parallel requests for faster scraping
4. **Rate Limiting**: Respectful delays between requests to avoid being blocked

## 1. Setup & Dependencies
First, let's install and import the required libraries.

In [ ]:
# Install required packages
!pip install requests beautifulsoup4 pandas tqdm

In [ ]:
# Import libraries
import requests
import json
import time
import random
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field, asdict
from typing import List, Optional, Dict, Any
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
from tqdm.notebook import tqdm
from IPython.display import display, HTML

print("✅ All libraries imported successfully!")

## 2. Data Models
Define data classes to structure our scraped data.

In [ ]:
@dataclass
class Founder:
    """Represents a startup founder"""
    name: str
    linkedin_url: Optional[str] = None
    title: Optional[str] = None


@dataclass
class Startup:
    """Represents a YC startup with all required fields"""
    company_name: str
    batch: str
    short_description: str
    founders: List[Founder] = field(default_factory=list)
    company_url: Optional[str] = None
    website: Optional[str] = None
    
    @property
    def founder_names(self) -> str:
        """Returns comma-separated founder names"""
        return ", ".join([f.name for f in self.founders])
    
    @property
    def founder_linkedin_urls(self) -> str:
        """Returns comma-separated founder LinkedIn URLs"""
        urls = [f.linkedin_url for f in self.founders if f.linkedin_url]
        return ", ".join(urls)
    
    def to_dict(self) -> Dict[str, str]:
        """Convert to dictionary for DataFrame"""
        return {
            'Company Name': self.company_name,
            'Batch': self.batch,
            'Short Description': self.short_description,
            'Founder Name(s)': self.founder_names,
            'Founder LinkedIn URL(s)': self.founder_linkedin_urls,
            'Company URL': self.company_url or '',
            'Website': self.website or ''
        }

print("✅ Data models defined!")

## 3. Algolia API Client
YC uses Algolia for their company search. We'll use their public API to efficiently fetch company listings.

In [ ]:
class YCAlgoliaClient:
    """
    Client for YC's Algolia search API.
    YC uses Algolia to power their company search functionality.
    """
    
    # Algolia API credentials (public, used by YC website frontend)
    ALGOLIA_APP_ID = "45BWZJ1SGC"
    ALGOLIA_API_KEY = "MjBjYjRiMzY0NzdhZWY0NjExY2NhZjYxMGIxYjc2MTAwNWFkNTkwNTc4NjgxYjU0YzFhYTY2ZGQ5OGY5NDMxZnJlc3RyaWN0SW5kaWNlcz0lNUIlMjJZQ0NvbXBhbnlfcHJvZHVjdGlvbiUyMiUyQyUyMllDQ29tcGFueV9CeV9MYXVuY2hfRGF0ZV9wcm9kdWN0aW9uJTIyJTVEJnRhZ0ZpbHRlcnM9JTVCJTIyeWNkY19wdWJsaWMlMjIlNUQmYW5hbHl0aWNzVGFncz0lNUIlMjJ5Y2RjJTIyJTVE"
    ALGOLIA_INDEX = "YCCompany_production"
    
    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
            'Accept': 'application/json',
            'Referer': 'https://www.ycombinator.com/companies',
        })
        self.base_url = f"https://{self.ALGOLIA_APP_ID}-dsn.algolia.net/1/indexes/{self.ALGOLIA_INDEX}/query"
    
    def search_companies(self, page: int = 0, hits_per_page: int = 100) -> Dict[str, Any]:
        """
        Search YC companies using Algolia API.
        """
        headers = {
            'x-algolia-application-id': self.ALGOLIA_APP_ID,
            'x-algolia-api-key': self.ALGOLIA_API_KEY,
            'Content-Type': 'application/json'
        }
        
        payload = {
            "query": "",
            "page": page,
            "hitsPerPage": hits_per_page,
            "attributesToRetrieve": [
                "name", "slug", "one_liner", "batch", "website",
                "all_locations", "team_size", "industries", "status"
            ]
        }
        
        try:
            response = self.session.post(
                self.base_url,
                headers=headers,
                json=payload,
                timeout=30
            )
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            print(f"❌ Algolia API error: {e}")
            return {"hits": [], "nbHits": 0}
    
    def get_all_companies(self, max_companies: int = 500) -> List[Dict[str, Any]]:
        """
        Fetch all companies up to max_companies limit.
        """
        all_companies = []
        page = 0
        hits_per_page = 100
        
        pbar = tqdm(total=max_companies, desc="Fetching companies")
        
        while len(all_companies) < max_companies:
            result = self.search_companies(page=page, hits_per_page=hits_per_page)
            hits = result.get('hits', [])
            
            if not hits:
                break
            
            all_companies.extend(hits)
            pbar.update(len(hits))
            
            total_hits = result.get('nbHits', 0)
            if (page + 1) * hits_per_page >= total_hits:
                break
            
            page += 1
            time.sleep(0.3)  # Rate limiting
        
        pbar.close()
        return all_companies[:max_companies]

print("✅ Algolia client class defined!")

## 4. Page Scraper for Founder Details
This class handles scraping individual company pages to extract founder information and LinkedIn URLs.

In [ ]:
class YCPageScraper:
    """
    Scrapes individual YC company pages for detailed founder information.
    """
    
    BASE_URL = "https://www.ycombinator.com/companies/"
    
    def __init__(self, max_workers: int = 5):
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.5',
        })
        self.max_workers = max_workers
        self.cache = {}
    
    def get_company_page(self, slug: str, retries: int = 3) -> Optional[str]:
        """Fetch a company's page HTML with retry logic."""
        if slug in self.cache:
            return self.cache[slug]
        
        url = urljoin(self.BASE_URL, slug)
        
        for attempt in range(retries):
            try:
                time.sleep(random.uniform(0.3, 0.8))
                response = self.session.get(url, timeout=30)
                response.raise_for_status()
                self.cache[slug] = response.text
                return response.text
            except requests.exceptions.RequestException as e:
                if attempt < retries - 1:
                    time.sleep(2 ** attempt)
        return None
    
    def extract_founders_from_html(self, html: str) -> List[Founder]:
        """Extract founder information from company page HTML."""
        founders = []
        soup = BeautifulSoup(html, 'html.parser')
        
        # Method 1: Parse Next.js data (most reliable)
        next_data_script = soup.find('script', {'id': '__NEXT_DATA__'})
        if next_data_script:
            try:
                data = json.loads(next_data_script.string)
                props = data.get('props', {}).get('pageProps', {})
                company_data = props.get('company', {})
                
                founder_list = company_data.get('founders', [])
                for f in founder_list:
                    founder = Founder(
                        name=f.get('full_name', f.get('name', 'Unknown')),
                        linkedin_url=f.get('linkedin_url'),
                        title=f.get('title')
                    )
                    founders.append(founder)
                    
                if founders:
                    return founders
            except (json.JSONDecodeError, KeyError, TypeError):
                pass
        
        # Method 2: Parse HTML directly for LinkedIn links
        linkedin_pattern = re.compile(r'linkedin\.com/in/([^/\s"\']+)')
        linkedin_links = soup.find_all('a', href=linkedin_pattern)
        
        for link in linkedin_links:
            href = link.get('href', '')
            if 'linkedin.com' in href:
                parent = link.find_parent(['div', 'section', 'article'])
                if parent:
                    text_content = parent.get_text(strip=True)
                    # Extract name (usually first text element)
                    name_match = re.match(r'^([A-Z][a-z]+ [A-Z][a-z]+)', text_content)
                    if name_match:
                        name = name_match.group(1)
                        existing_urls = [f.linkedin_url for f in founders]
                        if href not in existing_urls:
                            founders.append(Founder(name=name, linkedin_url=href))
        
        return founders
    
    def scrape_company(self, company_data: Dict[str, Any]) -> Optional[Startup]:
        """Scrape a single company's full information."""
        slug = company_data.get('slug', '')
        if not slug:
            return None
        
        startup = Startup(
            company_name=company_data.get('name', 'Unknown'),
            batch=company_data.get('batch', 'Unknown'),
            short_description=company_data.get('one_liner', ''),
            company_url=f"https://www.ycombinator.com/companies/{slug}",
            website=company_data.get('website', '')
        )
        
        html = self.get_company_page(slug)
        if html:
            founders = self.extract_founders_from_html(html)
            startup.founders = founders
        
        return startup
    
    def scrape_companies_parallel(self, companies: List[Dict[str, Any]]) -> List[Startup]:
        """Scrape multiple companies in parallel."""
        startups = []
        total = len(companies)
        
        with tqdm(total=total, desc="Scraping founder details") as pbar:
            with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
                future_to_company = {
                    executor.submit(self.scrape_company, company): company 
                    for company in companies
                }
                
                for future in as_completed(future_to_company):
                    try:
                        startup = future.result()
                        if startup:
                            startups.append(startup)
                    except Exception as e:
                        pass
                    pbar.update(1)
        
        return startups

print("✅ Page scraper class defined!")

## 5. Execute Scraping
Now let's run the scraper to collect data for 500 YC startups.

In [ ]:
# Configuration
MAX_COMPANIES = 500  # Number of startups to scrape
MAX_WORKERS = 5      # Concurrent requests (be respectful!)

print("🚀 Starting Y Combinator Startup Scraper")
print("=" * 50)
start_time = time.time()

In [ ]:
# Step 1: Fetch company listings from Algolia API
print("\n📥 Step 1: Fetching company listings from YC directory...")

algolia_client = YCAlgoliaClient()
companies = algolia_client.get_all_companies(max_companies=MAX_COMPANIES)

print(f"\n✅ Found {len(companies)} companies!")

In [ ]:
# Preview the raw data structure
print("\n📋 Sample company data structure:")
if companies:
    sample = companies[0]
    print(json.dumps(sample, indent=2))

In [ ]:
# Step 2: Scrape founder details for each company
print(f"\n🔍 Step 2: Scraping founder details for {len(companies)} companies...")
print("(This will take several minutes due to rate limiting)")

scraper = YCPageScraper(max_workers=MAX_WORKERS)
startups = scraper.scrape_companies_parallel(companies)

print(f"\n✅ Successfully scraped {len(startups)} companies!")

## 6. Data Analysis & Export
Let's analyze the scraped data and export it to CSV.

In [ ]:
# Convert to DataFrame
data = [startup.to_dict() for startup in startups]
df = pd.DataFrame(data)

print(f"\n📊 DataFrame created with {len(df)} rows")
print(f"\nColumn names:")
for col in df.columns:
    print(f"  - {col}")

In [ ]:
# Display first 10 companies
print("\n🔝 Top 10 Scraped Companies:")
display(df.head(10))

In [ ]:
# Data quality statistics
print("\n📈 Data Quality Statistics:")
print("=" * 40)
print(f"Total companies scraped: {len(df)}")
print(f"Companies with founders: {df['Founder Name(s)'].apply(lambda x: len(x) > 0).sum()}")
print(f"Companies with LinkedIn URLs: {df['Founder LinkedIn URL(s)'].apply(lambda x: len(x) > 0).sum()}")
print(f"Companies with descriptions: {df['Short Description'].apply(lambda x: len(str(x)) > 0).sum()}")

# Batch distribution
print(f"\n📅 Top 10 Batches:")
print(df['Batch'].value_counts().head(10))

In [ ]:
# Export to CSV
OUTPUT_FILE = "yc_startups_500.csv"
df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')
print(f"\n💾 Data exported to: {OUTPUT_FILE}")

# Also export to JSON
df.to_json("yc_startups_500.json", orient='records', indent=2)
print(f"💾 Data exported to: yc_startups_500.json")

In [ ]:
# Final summary
elapsed_time = time.time() - start_time

print("\n" + "=" * 50)
print("✅ SCRAPING COMPLETE!")
print("=" * 50)
print(f"\n📊 Summary:")
print(f"  • Total companies scraped: {len(df)}")
print(f"  • Companies with founder info: {df['Founder Name(s)'].apply(lambda x: len(x) > 0).sum()}")
print(f"  • Companies with LinkedIn URLs: {df['Founder LinkedIn URL(s)'].apply(lambda x: len(x) > 0).sum()}")
print(f"  • Time elapsed: {elapsed_time:.2f} seconds ({elapsed_time/60:.1f} minutes)")
print(f"\n📁 Output files:")
print(f"  • yc_startups_500.csv")
print(f"  • yc_startups_500.json")

## 7. Sample Output Preview
Let's view some sample entries from our scraped data.

In [ ]:
# Display sample entries with all details
print("\n📋 Sample Entries (with full details):")
print("=" * 60)

for idx, row in df.head(5).iterrows():
    print(f"\n🏢 {row['Company Name']}")
    print(f"   Batch: {row['Batch']}")
    print(f"   Description: {row['Short Description'][:100]}..." if len(str(row['Short Description'])) > 100 else f"   Description: {row['Short Description']}")
    print(f"   Founders: {row['Founder Name(s)'] if row['Founder Name(s)'] else 'N/A'}")
    print(f"   LinkedIn: {row['Founder LinkedIn URL(s)'][:80]}..." if len(str(row['Founder LinkedIn URL(s)'])) > 80 else f"   LinkedIn: {row['Founder LinkedIn URL(s)'] if row['Founder LinkedIn URL(s)'] else 'N/A'}")
    print("-" * 60)